Step 1: Import libraries needed for cleaning.

In [2]:
import pandas as pd
import glob
import os

Step 2: Combine all per-match info files into one long (type, key, value, match_id) table.

In [3]:
import glob

info_files = glob.glob('ipl_data/*_info.csv')
print(f"Found {len(info_files)} info files")

all_info_rows = []
for filepath in info_files:
    match_id = filepath.split('\\')[-1].split('/')[-1].replace('_info.csv', '')
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split(',', 2)
            if len(parts) == 3:
                parts.append(match_id)   # tag each row with which match it belongs to
                all_info_rows.append(parts)

match_info_long = pd.DataFrame(all_info_rows, columns=['type', 'key', 'value', 'match_id'])
print(match_info_long.shape)
match_info_long.head(10)

Found 1243 info files
(94447, 4)


,type,key,value,match_id
0,info,balls_per_over,6,1082591
1,info,team,Sunrisers Hyderabad,1082591
2,info,team,Royal Challengers Bangalore,1082591
3,info,team_type,club,1082591
4,info,gender,male,1082591
5,info,season,2017,1082591
6,info,date,2017/04/05,1082591
7,info,event,Indian Premier League,1082591
8,info,match_id,1082591,1082591
9,info,match_type,T20,1082591


Step 3: Pivot the single-value info keys (season, venue, winner, etc.) into one row per match.

In [4]:
# Step 2a: keys that appear ONCE per match — safe to pivot directly
single_value_keys = ['season', 'date', 'venue', 'city', 'toss_winner', 'toss_decision',
                      'winner', 'gender', 'match_type', 'event', 'balls_per_over']

single_df = match_info_long[match_info_long['key'].isin(single_value_keys)]

match_info_wide = single_df.pivot_table(
    index='match_id', columns='key', values='value', aggfunc='first'
).reset_index()

print(match_info_wide.shape)
match_info_wide.head()

(1243, 12)


key,match_id,balls_per_over,city,date,event,gender,match_type,season,toss_decision,toss_winner,venue,winner
0,1082591,6,Hyderabad,2017/04/05,Indian Premier League,male,T20,2017,field,Royal Challengers Bangalore,"""Rajiv Gandhi International Stadium, Uppal""",Sunrisers Hyderabad
1,1082592,6,Pune,2017/04/06,Indian Premier League,male,T20,2017,field,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,Rising Pune Supergiant
2,1082593,6,Rajkot,2017/04/07,Indian Premier League,male,T20,2017,field,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders
3,1082594,6,Indore,2017/04/08,Indian Premier League,male,T20,2017,field,Kings XI Punjab,Holkar Cricket Stadium,Kings XI Punjab
4,1082595,6,Bengaluru,2017/04/08,Indian Premier League,male,T20,2017,bat,Royal Challengers Bangalore,M.Chinnaswamy Stadium,Royal Challengers Bangalore


Step 4: Pivot the two 'team' rows per match into separate team1/team2 columns.

In [5]:
team_rows = match_info_long[match_info_long['key'] == 'team'].copy()

# number each team within a match: 1st occurrence = team1, 2nd = team2
team_rows['team_number'] = team_rows.groupby('match_id').cumcount() + 1

teams_wide = team_rows.pivot_table(
    index='match_id', columns='team_number', values='value', aggfunc='first'
).reset_index()

teams_wide.columns = ['match_id', 'team1', 'team2']

print(teams_wide.shape)
teams_wide.head()

(1243, 3)


,match_id,team1,team2
0,1082591,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1082592,Rising Pune Supergiant,Mumbai Indians
2,1082593,Gujarat Lions,Kolkata Knight Riders
3,1082594,Kings XI Punjab,Rising Pune Supergiant
4,1082595,Royal Challengers Bangalore,Delhi Daredevils


Merging the teams and the match details(pivot table)

In [6]:
match_info = match_info_wide.merge(teams_wide, on='match_id', how='left')

print(match_info.shape)
match_info.head()

(1243, 14)


,match_id,balls_per_over,city,date,event,gender,match_type,season,toss_decision,toss_winner,venue,winner,team1,team2
0,1082591,6,Hyderabad,2017/04/05,Indian Premier League,male,T20,2017,field,Royal Challengers Bangalore,"""Rajiv Gandhi International Stadium, Uppal""",Sunrisers Hyderabad,Sunrisers Hyderabad,Royal Challengers Bangalore
1,1082592,6,Pune,2017/04/06,Indian Premier League,male,T20,2017,field,Rising Pune Supergiant,Maharashtra Cricket Association Stadium,Rising Pune Supergiant,Rising Pune Supergiant,Mumbai Indians
2,1082593,6,Rajkot,2017/04/07,Indian Premier League,male,T20,2017,field,Kolkata Knight Riders,Saurashtra Cricket Association Stadium,Kolkata Knight Riders,Gujarat Lions,Kolkata Knight Riders
3,1082594,6,Indore,2017/04/08,Indian Premier League,male,T20,2017,field,Kings XI Punjab,Holkar Cricket Stadium,Kings XI Punjab,Kings XI Punjab,Rising Pune Supergiant
4,1082595,6,Bengaluru,2017/04/08,Indian Premier League,male,T20,2017,bat,Royal Challengers Bangalore,M.Chinnaswamy Stadium,Royal Challengers Bangalore,Royal Challengers Bangalore,Delhi Daredevils


Step 6: Sanity-check match_info for missing values and confirm the row count.

In [7]:
print(match_info['match_id'].nunique())     # should be 1243
print(match_info.isnull().sum())            # see which columns have gaps (e.g. 'city' might be missing for some venues)

1243
match_id           0
balls_per_over     0
city               0
date               0
event              0
gender             0
match_type         0
season             0
toss_decision      0
toss_winner        0
venue              0
winner            25
team1              0
team2              0
dtype: int64


Step 7: Inspect the matches with no recorded winner (ties/no-results).

In [8]:
# see which matches have no winner and why
no_winner = match_info[match_info['winner'].isnull()]
print(no_winner[['match_id', 'season', 'team1', 'team2']])

     match_id   season                        team1  \
33    1082625     2017                Gujarat Lions   
128   1175365     2019        Kolkata Knight Riders   
167   1178424     2019  Royal Challengers Bangalore   
169   1178426     2019               Mumbai Indians   
180   1216493  2020/21               Delhi Capitals   
199   1216512  2020/21        Kolkata Knight Riders   
204   1216517  2020/21               Mumbai Indians   
234   1216547  2020/21  Royal Challengers Bangalore   
258   1254077     2021               Delhi Capitals   
417   1359519     2023         Lucknow Super Giants   
549   1473469     2025               Delhi Capitals   
561   1473481     2025                 Punjab Kings   
572   1473492     2025               Delhi Capitals   
575   1473495     2025                 Punjab Kings   
603   1527685     2026        Kolkata Knight Riders   
629   1529281     2026        Kolkata Knight Riders   
732    392190     2009        Kolkata Knight Riders   
796    419

Fixing of the team names with new names

In [9]:
team_name_fix = {
    'Delhi Daredevils': 'Delhi Capitals',
    'Kings XI Punjab': 'Punjab Kings',
    'Rising Pune Supergiant': 'Rising Pune Supergiants',
    'Royal Challengers Bangalore': 'Royal Challengers Bengaluru'
}

for col in ['team1', 'team2', 'winner']:
    match_info[col] = match_info[col].replace(team_name_fix)

# verify — should now show fewer unique names
print(sorted(match_info['team1'].unique()))

['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']


Step 9: Save the cleaned match_info table to disk.

In [10]:
import os
os.makedirs('data/processed', exist_ok=True)
match_info.to_csv('data/processed/match_info.csv', index=False)
print("Saved:", match_info.shape)

Saved: (1243, 14)


Step 10: Reload all_matches, fix the season data type, and apply the same team-name standardization.

In [11]:
all_matches = pd.read_csv('ipl_data/all_matches.csv')

# season mixed-type issue
all_matches['season'] = all_matches['season'].astype(str)

# apply the same team name standardization
for col in ['batting_team', 'bowling_team']:
    all_matches[col] = all_matches[col].replace(team_name_fix)

# verify
print(sorted(all_matches['batting_team'].unique()))
print(all_matches.dtypes['season'])

C:\Users\ramha.LAPTOP-AB00D4P2\AppData\Local\Temp\ipykernel_7184\2727367495.py:1: DtypeWarning: Columns (0: season, 1: non_boundary, 2: fielder_3) have mixed types. Specify dtype option on import or set low_memory=False.
  all_matches = pd.read_csv('ipl_data/all_matches.csv')


['Chennai Super Kings', 'Deccan Chargers', 'Delhi Capitals', 'Gujarat Lions', 'Gujarat Titans', 'Kochi Tuskers Kerala', 'Kolkata Knight Riders', 'Lucknow Super Giants', 'Mumbai Indians', 'Pune Warriors', 'Punjab Kings', 'Rajasthan Royals', 'Rising Pune Supergiants', 'Royal Challengers Bengaluru', 'Sunrisers Hyderabad']
str


Step 11: Check for fully duplicate rows and duplicate (match_id, innings, ball) keys.

In [12]:
print("Duplicate rows:", all_matches.duplicated().sum())
print("Duplicate (match_id, innings, ball) combos:", 
      all_matches.duplicated(subset=['match_id', 'innings', 'ball']).sum())

Duplicate rows: 0
Duplicate (match_id, innings, ball) combos: 42


Step 12: Inspect a sample of the duplicate rows keyed on 'ball'.

In [13]:
dupe_keys = all_matches[all_matches.duplicated(subset=['match_id', 'innings', 'ball'], keep=False)]
print(dupe_keys.shape)
dupe_keys.sort_values(['match_id', 'innings', 'ball']).head(20)

(84, 27)


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
164765,336005,2007/08,2008-05-04,Sawai Mansingh Stadium,2,6.1,6.1,Rajasthan Royals,Chennai Super Kings,GC Smith,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
164774,336005,2007/08,2008-05-04,Sawai Mansingh Stadium,2,6.1,6.6,Rajasthan Royals,Chennai Super Kings,GC Smith,...,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168931,336024,2007/08,2008-05-18,"Rajiv Gandhi International Stadium, Uppal",1,18.1,18.1,Mumbai Indians,Deccan Chargers,PR Shah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
168940,336024,2007/08,2008-05-18,"Rajiv Gandhi International Stadium, Uppal",1,18.1,18.6,Mumbai Indians,Deccan Chargers,PR Shah,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194867,419141,2009/10,2010-04-05,"Vidarbha Cricket Association Stadium, Jamtha",2,1.1,1.1,Deccan Chargers,Rajasthan Royals,AC Gilchrist,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
194876,419141,2009/10,2010-04-05,"Vidarbha Cricket Association Stadium, Jamtha",2,1.1,1.6,Deccan Chargers,Rajasthan Royals,AC Gilchrist,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197441,419152,2009/10,2010-04-13,Brabourne Stadium,1,3.1,3.1,Mumbai Indians,Delhi Capitals,SR Tendulkar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
197450,419152,2009/10,2010-04-13,Brabourne Stadium,1,3.1,3.6,Mumbai Indians,Delhi Capitals,SR Tendulkar,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
224477,548334,2012,2012-04-22,Wankhede Stadium,1,19.1,19.1,Mumbai Indians,Punjab Kings,NLTC Perera,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
224486,548334,2012,2012-04-22,Wankhede Stadium,1,19.1,19.6,Mumbai Indians,Punjab Kings,NLTC Perera,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Step 13: Re-check duplicates using the actual_delivery column instead of ball.

In [14]:
print("Duplicate (match_id, innings, actual_delivery) combos:", 
      all_matches.duplicated(subset=['match_id', 'innings', 'actual_delivery']).sum())

Duplicate (match_id, innings, actual_delivery) combos: 11075


Step 14: Inspect a sample of the duplicates keyed on actual_delivery.

In [15]:
dupe_keys2 = all_matches[all_matches.duplicated(subset=['match_id', 'innings', 'actual_delivery'], keep=False)]
print(dupe_keys2.shape)
print(dupe_keys2['match_id'].nunique(), "matches affected")
dupe_keys2.sort_values(['match_id', 'innings', 'actual_delivery']).head(20)

(21493, 27)
1240 matches affected


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
159136,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,0.3,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159137,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,0.4,0.3,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159153,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,3.1,3.1,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159154,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,3.2,3.1,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159212,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,12.5,12.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159213,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,12.6,12.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159225,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,14.5,14.5,Kolkata Knight Riders,Royal Challengers Bengaluru,BB McCullum,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159226,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,1,14.6,14.5,Kolkata Knight Riders,Royal Challengers Bengaluru,DJ Hussey,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159259,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,2,0.2,0.2,Royal Challengers Bengaluru,Kolkata Knight Riders,W Jaffer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
159260,335982,2007/08,2008-04-18,M Chinnaswamy Stadium,2,0.3,0.2,Royal Challengers Bengaluru,Kolkata Knight Riders,W Jaffer,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


Step 15: Add a guaranteed-unique per-innings delivery sequence number.

In [16]:
all_matches['delivery_seq'] = all_matches.groupby(['match_id', 'innings']).cumcount() + 1

Step 16: Save the fully cleaned all_matches table to disk.

In [17]:
all_matches.to_csv('data/processed/all_matches_clean.csv', index=False)
print("Saved:", all_matches.shape)

Saved: (295732, 28)


In [18]:
print(all_matches[all_matches['striker'].str.contains('kohli', case=False, na=False)]['striker'].unique())

<ArrowStringArray>
['V Kohli', 'T Kohli']
Length: 2, dtype: str
